# **Microscopy RAW Dataset Analysis**

Load, process, and analyse RAW images from the Microscopy dataset.

The RAW images are single-channel Bayer mosaics (RGGB pattern, 256x256). In the current export they are stored as float TIFF files with an effective 16-bit intensity range. An ISP (Image Signal Processing) pipeline converts them to RGB via:

> Black-level removal -> Demosaicing -> White Balance -> Colour Correction -> Sharpening -> Denoising -> Gamma Correction

The annotation file also contains `UNC` samples. 
`UNC` stands for unclassifiable: it is inspected below as a quality-control bucket, but it is excluded from supervised classification because it does not represent a coherent morphology class. The residual image-correlation analysis below shows that the present `UNC` images do not form a correlated visual group.

1. Dataset loading & annotation parsing
2. Class filtering and missing-file handling
3. Class distribution & imbalance analysis
4. `UNC` quality-control introspection
5. RAW image statistics (intensity histograms, per-class statistics)
6. ISP pipeline walkthrough - visual comparison of each stage
7. Comparison of different ISP configurations (debayer / sharpening / denoising)
8. Per-channel statistics after ISP processing
9. PyTorch Dataset & DataLoader with stratified split
10. Sample visualization per class
11. Batch-level quality checks


In [ ]:
import os
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import tifffile
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
from torch.utils.data import Dataset, DataLoader
from dotenv import load_dotenv

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root."""
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').is_dir() and (candidate / 'requirements.txt').is_file():
            return candidate
    return start

In [ ]:
PROJECT_ROOT = str(find_project_root(Path.cwd().resolve()))

# Load environment variables from .env
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

# ---------- CONFIGURATION ----------
ROOT_DIR = os.environ['MICROSCOPY_ROOT']
INTENSITY_RATIO = 1.0               # scale factor (only 1.0 supported for raw)
MIN_SAMPLES_PER_CLASS = 15          # match the model-comparison training filter
BELOW_THRESHOLD_STRATEGY = 'drop'   # 'drop' or 'merge_to_rare'
RARE_CLASS_NAME = 'RARE'            # merged rare class name (if merge_to_rare)
EXCLUDE_CLASSES = {'UNC'}           # Unclassifiable bucket, not a classification target
MISSING_IMAGE_POLICY = 'drop'       # drop annotations whose files are absent after duplicate cleanup
BITS = 16                           # effective raw image bit depth used for normalisation

print(f"Project root  : {PROJECT_ROOT}")
print(f"Microscopy data directory: {ROOT_DIR}")
print(f"Max pixel value for normalisation: {2**BITS - 1}")
print(f"Excluded from classification: {sorted(EXCLUDE_CLASSES)}")
print(f"Missing-image policy: {MISSING_IMAGE_POLICY}")


In [ ]:
# --- Figure export setup for LaTeX report ---
FIGURES_DIR = Path(PROJECT_ROOT) / 'report' / 'figures' / 'microscopy'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REPORT_FIG_DPI = 180
REPORT_FIG_MAX_PIXELS = 3_500_000
REPORT_FIG_PALETTE_COLORS = 192
REPORT_FIG_COMPRESSION_LEVEL = 9

REPORT_PALETTE = [
    '#2F5D7C',  # deep blue
    '#3A9D8F',  # teal
    '#D99B2B',  # amber
    '#C95F47',  # vermilion
    '#7B5EA7',  # muted purple
    '#5E8C61',  # green
    '#6F7D8C',  # slate
    '#A66F3F',  # copper
]
REPORT_BLUE = REPORT_PALETTE[0]
REPORT_TEAL = REPORT_PALETTE[1]
REPORT_AMBER = REPORT_PALETTE[2]
REPORT_VERMILION = REPORT_PALETTE[3]
REPORT_PURPLE = REPORT_PALETTE[4]
REPORT_GREEN = REPORT_PALETTE[5]
REPORT_SLATE = REPORT_PALETTE[6]
REPORT_COPPER = REPORT_PALETTE[7]
REPORT_NEUTRAL = '#5F6C76'
REPORT_EDGE = '#24313A'
REPORT_GRID = '#D8DEE6'
REPORT_ACCENT = '#B23A48'
REPORT_HEATMAP_CMAP = 'YlGnBu'
REPORT_DIVERGING_CMAP = 'RdBu_r'
CLASS_COLORS = {
    'BAS': REPORT_BLUE,
    'EOS': REPORT_VERMILION,
    'KSC': REPORT_SLATE,
    'LYA': REPORT_PURPLE,
    'LYT': '#4C78A8',
    'MON': REPORT_TEAL,
    'NGB': REPORT_GREEN,
    'NGS': REPORT_AMBER,
    'RARE': REPORT_COPPER,
}
CHANNEL_COLORS = {
    'R': REPORT_VERMILION,
    'G': REPORT_TEAL,
    'G1': REPORT_TEAL,
    'G2': REPORT_GREEN,
    'B': REPORT_BLUE,
}


def colors_for_labels(labels):
    return [CLASS_COLORS.get(label, REPORT_PALETTE[idx % len(REPORT_PALETTE)]) for idx, label in enumerate(labels)]


plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': REPORT_FIG_DPI,
    'savefig.facecolor': 'white',
    'savefig.edgecolor': 'white',
    'axes.titlesize': 17,
    'axes.titleweight': 'bold',
    'axes.labelsize': 15,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 12,
    'legend.title_fontsize': 13,
    'figure.titlesize': 19,
    'font.size': 13,
    'axes.linewidth': 1.1,
    'lines.linewidth': 2.0,
    'legend.frameon': True,
    'legend.framealpha': 0.95,
    'axes.edgecolor': REPORT_EDGE,
    'grid.color': REPORT_GRID,
    'grid.alpha': 0.45,
})
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=REPORT_PALETTE)


def _optimize_png_for_latex(path, max_pixels=REPORT_FIG_MAX_PIXELS):
    """Keep LaTeX figures compact while preserving readable text."""
    try:
        from PIL import Image
    except ImportError:
        return

    path = Path(path)
    with Image.open(path) as img:
        if img.mode in {'RGBA', 'LA'}:
            background = Image.new('RGBA', img.size, (255, 255, 255, 255))
            background.alpha_composite(img.convert('RGBA'))
            img = background.convert('RGB')
        elif img.mode not in {'RGB', 'L', 'P'}:
            img = img.convert('RGB')

        pixels = img.width * img.height
        if pixels > max_pixels:
            scale = (max_pixels / pixels) ** 0.5
            new_size = (max(1, int(img.width * scale)), max(1, int(img.height * scale)))
            img = img.resize(new_size, Image.Resampling.LANCZOS)

        if img.mode == 'RGB':
            img = img.quantize(
                colors=REPORT_FIG_PALETTE_COLORS,
                method=Image.Quantize.MEDIANCUT,
                dither=Image.Dither.FLOYDSTEINBERG,
            )

        img.save(path, format='PNG', optimize=True, compress_level=REPORT_FIG_COMPRESSION_LEVEL)


def savefig(fig, name, dpi=REPORT_FIG_DPI):
    """Save a compact, white-background PNG for the LaTeX report."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(
        path,
        format='png',
        dpi=dpi,
        bbox_inches='tight',
        pad_inches=0.08,
        facecolor='white',
        transparent=False,
    )
    _optimize_png_for_latex(path)
    size_kib = path.stat().st_size / 1024
    print(f"  Saved: {path.name} ({size_kib:.0f} KiB)")


print(f"Figures will be saved to: {FIGURES_DIR}")


## 1. Load annotations and RAW image paths

In [ ]:
IMAGE_FILE_TYPES = {'dng', 'png', 'tif', 'tiff'}

VALID_CLASSES = [
    'BAS', 'EBO', 'EOS', 'KSC', 'LYA', 'LYT', 'MMZ', 'MOB',
    'MON', 'MYB', 'MYO', 'NGB', 'NGS', 'PMB', 'PMO'
]
INSPECTION_ONLY_CLASSES = ['UNC']

CLASS_FULL_NAMES = {
    'BAS': 'Basophil',                      'EBO': 'Erythroblast',
    'EOS': 'Eosinophil',                    'KSC': 'Smudge cell',
    'LYA': 'Lymphocyte (atypical)',         'LYT': 'Lymphocyte (typical)',
    'MMZ': 'Meta-myelocyte',                'MOB': 'Monoblast',
    'MON': 'Monocyte',                      'MYB': 'Myeloblast',
    'MYO': 'Myelocyte',                     'NGB': 'Neutrophil (band)',
    'NGS': 'Neutrophil (segmented)',        'PMB': 'Pro-myelocyte (bilobed)',
    'PMO': 'Pro-monocyte',                  'UNC': 'Unclassifiable',
}

# Camera parameters
BLACK_LEVEL = [9.834368023181512e-06] * 4
WHITE_BALANCE = [-0.6567, 1.9673, 3.5304]
COLOUR_MATRIX = [-2.0338, 0.0933, 0.4157,
                 -0.0286, 2.6464, -0.0574,
                 -0.5516, -0.0947, 2.9308]
CAMERA_PARAMETERS = (BLACK_LEVEL, WHITE_BALANCE, COLOUR_MATRIX)

# Dataset-level normalisation stats (raw after ISP)
DATASET_MEAN = [0.91, 0.83, 0.94]
DATASET_STD  = [0.08, 0.13, 0.05]

INVALID_FILES = {'Ma190c_lame3_zone13_composite_Mcropped_2.tiff'}

raw_dir = os.path.join(ROOT_DIR, 'images', f'raw_scale{int(INTENSITY_RATIO * 100):03d}')
label_file = os.path.join(ROOT_DIR, 'labels', 'Ma190c_annotations.dat')

print(f"RAW folder  : {raw_dir}")
print(f"Label file  : {label_file}")
print(f"\nCamera parameters:")
print(f"  Black level    : {BLACK_LEVEL[0]:.6e}")
print(f"  White balance  : {WHITE_BALANCE}")
print(f"  Colour matrix  : (3x3)")


In [ ]:
def read_label_file(label_file_path: str):
    """Parse the Ma190c_annotations.dat annotation file."""
    images, class_labels = [], []
    with open(label_file_path, 'rb') as f:
        for line in f:
            line_decoded = line.decode('utf-8').strip()
            if not line_decoded:
                continue
            file_name, class_label = line_decoded.split()
            images.append(f"{file_name}.tiff")
            class_labels.append(class_label)
    return images, class_labels

images_filenames, class_labels_raw = read_label_file(label_file)
print(f"Total annotations : {len(images_filenames)}")
print(f"Unique classes    : {len(set(class_labels_raw))}")
print(f"Classes found     : {sorted(set(class_labels_raw))}")

## 2. Class filtering and dataset construction

In [ ]:
# Per-class counts (before filtering)
label_counts = Counter(class_labels_raw)
observed_classes = sorted(label_counts)

# Keep only supported classification classes actually present in the annotations.
# QC-only labels such as UNC are intentionally excluded before thresholding.
valid_classes = [
    cls for cls in VALID_CLASSES
    if label_counts.get(cls, 0) > 0 and cls not in EXCLUDE_CLASSES
]
inspection_counts = {cls: label_counts.get(cls, 0) for cls in INSPECTION_ONLY_CLASSES if label_counts.get(cls, 0) > 0}
unsupported_classes = [
    cls for cls in observed_classes
    if cls not in VALID_CLASSES and cls not in INSPECTION_ONLY_CLASSES
]

# Apply min_samples_per_class threshold
eligible_classes = [cls for cls in valid_classes if label_counts[cls] >= MIN_SAMPLES_PER_CLASS]
low_count_classes = [cls for cls in valid_classes if label_counts[cls] < MIN_SAMPLES_PER_CLASS]

# Build class mapping based on the chosen strategy
class_mapping = {}
if BELOW_THRESHOLD_STRATEGY == 'merge_to_rare':
    for cls in eligible_classes:
        class_mapping[cls] = cls
    for cls in low_count_classes:
        class_mapping[cls] = RARE_CLASS_NAME
    selected_classes = set(eligible_classes + low_count_classes)
else:  # 'drop'
    for cls in eligible_classes:
        class_mapping[cls] = cls
    selected_classes = set(eligible_classes)

print(f"Observed annotation classes: {observed_classes}")
print(f"Inspection-only classes: {inspection_counts}")
print(f"Unsupported labels ignored: {unsupported_classes}")
print(f"Strategy: below_threshold = '{BELOW_THRESHOLD_STRATEGY}', min_samples = {MIN_SAMPLES_PER_CLASS}")
print(f"Eligible classification classes (>= {MIN_SAMPLES_PER_CLASS} annotations): {eligible_classes}")
print(f"Below-threshold classes (dropped/merged): {low_count_classes}")
print(f"Selected classification classes: {sorted(selected_classes)}")


In [ ]:
# Filter out invalid files, QC-only labels, unsupported/low-count classes, and missing images.
filtered_data = []
missing_data = []

for img, cls in zip(images_filenames, class_labels_raw):
    if cls in EXCLUDE_CLASSES or cls not in selected_classes or img in INVALID_FILES:
        continue

    path = os.path.join(raw_dir, img)
    if not os.path.isfile(path):
        missing_data.append((img, cls))
        continue

    filtered_data.append((img, class_mapping[cls], path))

if missing_data and MISSING_IMAGE_POLICY == 'error':
    preview = ', '.join(img for img, _ in missing_data[:5])
    suffix = '...' if len(missing_data) > 5 else ''
    raise FileNotFoundError(f"{len(missing_data)} selected annotations have no RAW file: {preview}{suffix}")
if not filtered_data:
    raise RuntimeError('No valid images remaining after filtering.')

final_images = [img for img, _, _ in filtered_data]
final_labels = [lbl for _, lbl, _ in filtered_data]
image_paths = [path for _, _, path in filtered_data]

# Create class-to-index mapping
classes = sorted(set(final_labels))
if RARE_CLASS_NAME in classes:
    classes = [c for c in classes if c != RARE_CLASS_NAME] + [RARE_CLASS_NAME]
class_to_idx = {cls: i for i, cls in enumerate(classes)}
labels = [class_to_idx[lbl] for lbl in final_labels]

print(f"Total images after filtering: {len(image_paths)}")
print(f"Dropped selected annotations with missing RAW file: {len(missing_data)}")
if missing_data:
    print(f"Missing selected annotations by class: {dict(sorted(Counter(cls for _, cls in missing_data).items()))}")
print(f"Final classification classes ({len(classes):>2}): {classes}")
print(f"\nClass -> Index mapping:")
for cls, idx in class_to_idx.items():
    full = CLASS_FULL_NAMES.get(cls, cls)
    print(f"  {idx:>2d} : {cls} - {full}")


## 3. Class distribution & imbalance analysis

In [ ]:
label_str = [classes[l] for l in labels]
counts = Counter(label_str)
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)

total = sum(c for _, c in sorted_counts)
n_classes = len(classes)
ideal_balanced = total / n_classes

# --- Plot: class distribution + imbalance ratio ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Bar chart
ax = axes[0]
class_names_sorted = [c[0] for c in sorted_counts]
class_values = [c[1] for c in sorted_counts]
colors = colors_for_labels(class_names_sorted)
bars = ax.bar(class_names_sorted, class_values, color=colors, edgecolor=REPORT_EDGE, linewidth=0.5)
ax.axhline(ideal_balanced, ls='--', color=REPORT_ACCENT, alpha=0.8, label=f'Balanced = {ideal_balanced:.0f}')
ax.set_xlabel('Class', fontsize=14)
ax.set_ylabel('Number of samples', fontsize=14)
ax.set_title('Class Distribution - Microscopy RAW', fontsize=18)
ax.tick_params(axis='x', rotation=35, labelsize=12)
ax.tick_params(axis='y', labelsize=12)
ax.legend(fontsize=11)
for bar, val in zip(bars, class_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(val), ha='center', va='bottom', fontsize=11)

# Imbalance ratio (each class count / max class count)
ax2 = axes[1]
max_count = max(class_values)
ratios = [v / max_count for v in class_values]
ax2.barh(class_names_sorted, ratios, color=colors, edgecolor=REPORT_EDGE, linewidth=0.5)
ax2.set_xlabel('Ratio to majority class', fontsize=14)
ax2.set_title('Imbalance Ratio (1.0 = majority class)', fontsize=18)
ax2.invert_yaxis()
for i, (r, v) in enumerate(zip(ratios, class_values)):
    ax2.text(r + 0.01, i, f'{r:.2f}  (n={v})', va='center', fontsize=11)
ax2.set_xlim(0, 1.3)

plt.tight_layout()
savefig(fig, 'class_distribution')
plt.show()

# --- Summary statistics ---
print(f"\n{'Class':>5}  {'Full Name':<30}  {'Count':>6}  {'%':>7}  {'Ratio':>6}")
print('-' * 65)
for name, count in sorted_counts:
    full = CLASS_FULL_NAMES.get(name, name)
    pct = 100 * count / total
    ratio = count / max_count
    print(f"{name:>5}  {full:<30}  {count:>6}  {pct:>6.1f}%  {ratio:>6.2f}")
print('-' * 65)
print(f"{'Total':>5}  {'':30}  {total:>6}")
print(f"\nImbalance ratio (max/min): {max_count / min(class_values):.1f}x")


## 4. RAW image statistics - intensity histograms & spatial analysis

In [ ]:
def load_raw_image(path: str) -> np.ndarray:
    """Load a RAW TIFF image and return float32 array (H, W)."""
    raw = tifffile.imread(path)
    return np.array(raw, dtype=np.float32)

# --- Sample a few images and compute global RAW statistics ---
N_SAMPLE = min(200, len(image_paths))
rng = np.random.default_rng(42)
sample_idxs = rng.choice(len(image_paths), N_SAMPLE, replace=False)

raw_mins, raw_maxs, raw_means, raw_stds = [], [], [], []
sample_images_raw = {}

for i, idx in enumerate(sample_idxs):
    img = load_raw_image(image_paths[idx])
    raw_mins.append(img.min())
    raw_maxs.append(img.max())
    raw_means.append(img.mean())
    raw_stds.append(img.std())
    if i < 6:  # keep a few for visualization
        sample_images_raw[idx] = img

raw_mins = np.array(raw_mins)
raw_maxs = np.array(raw_maxs)
raw_means = np.array(raw_means)
raw_stds = np.array(raw_stds)

print(f"RAW image statistics (sampled {N_SAMPLE} images):")
print(f"  Shape        : {img.shape}")
print(f"  Dtype (disk) : {tifffile.imread(image_paths[0]).dtype}")
print(f"  Min pixel    : {raw_mins.min():.2f}  (avg {raw_mins.mean():.2f})")
print(f"  Max pixel    : {raw_maxs.max():.2f}  (avg {raw_maxs.mean():.2f})")
print(f"  Mean pixel   : {raw_means.mean():.2f} ± {raw_means.std():.2f}")
print(f"  Std pixel    : {raw_stds.mean():.2f} ± {raw_stds.std():.2f}")
print(f"  Effective DR : {np.log2(raw_maxs.mean() + 1):.1f} bits")


### UNC introspection (excluded from classification)

`UNC` is treated as an inspection-only label. It marks images that are unclassifiable rather than a biological morphology class. The check below keeps `UNC` out of `classes`, then analyses only the present `UNC` files to verify whether they form a coherent visual group.

The correlation is computed on downsampled RAW images after subtracting the sample mean image. This removes the shared acquisition/background structure that otherwise dominates raw pixel correlations, so the remaining Pearson values measure residual image-to-image similarity.


In [ ]:
def parse_lame_zone(filename: str):
    match = re.search(r'lame(\d+)_zone(\d+)', filename)
    if match is None:
        return None, None
    return int(match.group(1)), int(match.group(2))


unc_records = []
for img, cls in zip(images_filenames, class_labels_raw):
    if cls != 'UNC':
        continue
    path = os.path.join(raw_dir, img)
    lame, zone = parse_lame_zone(img)
    unc_records.append({
        'filename': img,
        'path': path,
        'present': os.path.isfile(path),
        'lame': lame,
        'zone': zone,
    })

unc_present = [record for record in unc_records if record['present']]
unc_missing = [record for record in unc_records if not record['present']]
slide_counts = Counter(record['lame'] for record in unc_present)
zone_counts = Counter((record['lame'], record['zone']) for record in unc_present)

print(f"UNC annotations total       : {len(unc_records)}")
print(f"UNC RAW files present       : {len(unc_present)}")
print(f"UNC RAW files missing       : {len(unc_missing)}")
print(f"Distinct slides among UNC   : {len(slide_counts)}")
print(f"Distinct lame/zone pairs    : {len(zone_counts)}")
print(f"Max UNC images per zone     : {max(zone_counts.values()) if zone_counts else 0}")
print(f"Slide distribution          : {dict(sorted(slide_counts.items()))}")


def downsample_mean(img: np.ndarray, factor: int = 8) -> np.ndarray:
    h = (img.shape[0] // factor) * factor
    w = (img.shape[1] // factor) * factor
    img = img[:h, :w]
    return img.reshape(h // factor, factor, w // factor, factor).mean(axis=(1, 3)).ravel()


def standardize_rows(x: np.ndarray) -> np.ndarray:
    x = x.astype(np.float64, copy=True)
    x -= x.mean(axis=1, keepdims=True)
    std = x.std(axis=1, keepdims=True)
    std[std == 0] = 1.0
    return x / std


def upper_pairs(indices) -> list[tuple[int, int]]:
    indices = list(indices)
    return [(indices[i], indices[j]) for i in range(len(indices)) for j in range(i + 1, len(indices))]


def pairwise_values(z: np.ndarray, pairs: list[tuple[int, int]]) -> np.ndarray:
    if not pairs:
        return np.array([], dtype=np.float64)
    denom = z.shape[1] - 1
    return np.array([(z[i] @ z[j]) / denom for i, j in pairs], dtype=np.float64)


def describe_corr(values: np.ndarray) -> dict:
    if len(values) == 0:
        return {'n': 0, 'mean': np.nan, 'median': np.nan, 'mean_abs': np.nan, 'p05': np.nan, 'p95': np.nan}
    return {
        'n': int(len(values)),
        'mean': float(np.mean(values)),
        'median': float(np.median(values)),
        'mean_abs': float(np.mean(np.abs(values))),
        'p05': float(np.percentile(values, 5)),
        'p95': float(np.percentile(values, 95)),
    }


rng = np.random.default_rng(42)
reference_indices = []
for cls_idx in range(len(classes)):
    idxs = np.flatnonzero(np.array(labels) == cls_idx)
    n_take = min(30, len(idxs))
    reference_indices.extend(rng.choice(idxs, size=n_take, replace=False).tolist())

feature_paths = [record['path'] for record in unc_present] + [image_paths[i] for i in reference_indices]
feature_labels = ['UNC'] * len(unc_present) + [classes[labels[i]] for i in reference_indices]
features = np.stack([downsample_mean(load_raw_image(path), factor=8) for path in feature_paths])

# Residual correlations: remove the shared mean image before row-wise standardisation.
residual_features = features - features.mean(axis=0, keepdims=True)
z = standardize_rows(residual_features)
feature_labels_np = np.array(feature_labels)

unc_idx = np.flatnonzero(feature_labels_np == 'UNC')
ref_idx = np.flatnonzero(feature_labels_np != 'UNC')

unc_corr = pairwise_values(z, upper_pairs(unc_idx))

same_class_pairs = []
for cls in classes:
    cls_idx = np.flatnonzero(feature_labels_np == cls)
    same_class_pairs.extend(upper_pairs(cls_idx))
same_class_corr = pairwise_values(z, same_class_pairs)

cross_pairs = []
for _ in range(5000):
    i, j = rng.choice(ref_idx, size=2, replace=False)
    if feature_labels[i] != feature_labels[j]:
        cross_pairs.append((int(i), int(j)))
cross_class_corr = pairwise_values(z, cross_pairs)

summary = {
    'UNC within-class residual': describe_corr(unc_corr),
    'Reference same-class residual': describe_corr(same_class_corr),
    'Reference cross-class residual': describe_corr(cross_class_corr),
}

print("\nResidual Pearson correlation summary")
print(f"{'Group':<32} {'Pairs':>7} {'Mean':>9} {'Median':>9} {'Mean |r|':>10} {'P05':>9} {'P95':>9}")
print('-' * 91)
for group, stats in summary.items():
    print(
        f"{group:<32} {stats['n']:>7d} {stats['mean']:>9.3f} "
        f"{stats['median']:>9.3f} {stats['mean_abs']:>10.3f} "
        f"{stats['p05']:>9.3f} {stats['p95']:>9.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for values, label, color in [
    (unc_corr, 'UNC within', REPORT_VERMILION),
    (same_class_corr, 'Reference same class', REPORT_BLUE),
    (cross_class_corr, 'Reference cross class', REPORT_TEAL),
]:
    if len(values):
        ax.hist(values, bins=40, alpha=0.45, density=True, label=label, color=color)
ax.axvline(0, color=REPORT_EDGE, linestyle='--', linewidth=1)
ax.set_xlabel('Residual Pearson r')
ax.set_ylabel('Density')
ax.set_title('Residual pairwise correlation distributions')
ax.legend()

ax = axes[1]
if len(unc_idx) >= 2:
    unc_z = z[unc_idx]
    unc_matrix = np.corrcoef(unc_z)
    im = ax.imshow(unc_matrix, vmin=-0.5, vmax=0.5, cmap=REPORT_DIVERGING_CMAP)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Residual Pearson r')
    ax.set_title('UNC residual correlation matrix')
    ax.set_xlabel('UNC image index')
    ax.set_ylabel('UNC image index')
else:
    ax.text(0.5, 0.5, 'Not enough UNC images', ha='center', va='center')
    ax.axis('off')

plt.suptitle('UNC does not form a coherent residual-correlation group', fontsize=18, y=1.02)
plt.tight_layout()
savefig(fig, 'unc_residual_correlation')
plt.show()


**Interpretation.** In the cleaned dataset there are 30 `UNC` annotations, 25 corresponding RAW files still present, and 5 missing files already removed as part of the corrupt/duplicate cleanup. The 25 present `UNC` images are spread across 11 slides and 25 distinct lame/zone pairs, so they are not a repeated acquisition subset.

After removing the shared mean image, the `UNC` residual pairwise correlations are centered near zero (median about `-0.01`, mean absolute correlation about `0.09`) and overlap the cross-class reference baseline. This supports treating `UNC` as an unclassifiable quality-control bucket rather than a supervised classification class. It is therefore excluded before class mapping, splitting, training, and evaluation.


### RAW intensity distribution (global + per-image spread)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1) Global histogram of raw pixel values (aggregated from samples)
ax = axes[0]
all_pixels = np.concatenate([load_raw_image(image_paths[idx]).ravel()
                              for idx in sample_idxs[:50]])  # use 50 images
ax.hist(all_pixels, bins=300, color=REPORT_NEUTRAL, edgecolor='none', alpha=0.8)
ax.set_xlabel('Raw pixel intensity')
ax.set_ylabel('Count')
ax.set_title('Global RAW intensity histogram')
ax.axvline(all_pixels.mean(), color=REPORT_ACCENT, ls='--', label=f'Mean={all_pixels.mean():.0f}')
ax.legend()

# 2) Per-image mean distribution
ax = axes[1]
ax.hist(raw_means, bins=40, color=REPORT_BLUE, edgecolor=REPORT_EDGE, alpha=0.8)
ax.set_xlabel('Per-image mean intensity')
ax.set_ylabel('Number of images')
ax.set_title('Distribution of per-image means')

# 3) Per-image std distribution
ax = axes[2]
ax.hist(raw_stds, bins=40, color=REPORT_VERMILION, edgecolor=REPORT_EDGE, alpha=0.8)
ax.set_xlabel('Per-image std intensity')
ax.set_ylabel('Number of images')
ax.set_title('Distribution of per-image stds')

plt.suptitle('RAW Bayer Image Statistics', fontsize=18, y=1.02)
plt.tight_layout()
savefig(fig, 'raw_intensity_histograms')
plt.show()


### Per-class RAW statistics

In [ ]:
# Compute per-class mean/std of raw images (sample up to 30 images per class)
per_class_means = {cls: [] for cls in classes}
per_class_stds = {cls: [] for cls in classes}

for idx in range(len(image_paths)):
    cls_name = classes[labels[idx]]
    if len(per_class_means[cls_name]) >= 30:
        continue
    img = load_raw_image(image_paths[idx])
    per_class_means[cls_name].append(img.mean())
    per_class_stds[cls_name].append(img.std())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot of per-class means
ax = axes[0]
data_means = [per_class_means[cls] for cls in classes]
bp = ax.boxplot(data_means, labels=classes, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_for_labels(classes)):
    patch.set_facecolor(color)
ax.set_xlabel('Class')
ax.set_ylabel('Mean raw intensity')
ax.set_title('Per-class RAW mean intensity')
ax.tick_params(axis='x', rotation=35, labelsize=12)
ax.tick_params(axis='y', labelsize=12)

# Box plot of per-class stds
ax = axes[1]
data_stds = [per_class_stds[cls] for cls in classes]
bp = ax.boxplot(data_stds, labels=classes, patch_artist=True)
for patch, color in zip(bp['boxes'], colors_for_labels(classes)):
    patch.set_facecolor(color)
ax.set_xlabel('Class')
ax.set_ylabel('Std raw intensity')
ax.set_title('Per-class RAW intensity spread')
ax.tick_params(axis='x', rotation=45)

plt.suptitle('Per-class RAW statistics (up to 30 images/class)', fontsize=18, y=1.02)
plt.tight_layout()
savefig(fig, 'per_class_raw_stats')
plt.show()


## 5. ISP Pipeline - stage-by-stage walkthrough

The RAW-to-RGB pipeline applies the following stages:

| Stage | Description |
|---|---|
| Black-level removal | Subtract per-channel black level from RGGB pattern |
| Demosaicing | Recover full RGB from Bayer CFA (bilinear / Malvar2004 / Menon2007) |
| White balance | Apply camera WB gains `[-0.66, 1.97, 3.53]` |
| Colour correction | 3x3 CCM multiplication |
| Dynamic range norm. | Clip to 99th percentile and normalize to [0, 1] |
| Sharpening | Unsharp masking or convolution filter (in YUV luma) |
| Denoising | Gaussian / Median / FFT / TV / Bilateral (in YUV luma) |
| Gamma correction | Power-law gamma = 2.2 |

In [ ]:
from colour_demosaicing import demosaicing_CFA_Bayer_bilinear

# Ensure scripts/ is in sys.path so we can import the processing package
SCRIPTS_DIR = os.path.join(PROJECT_ROOT, 'scripts')
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

from processing.numpy_pipeline import (
    RawProcessingPipeline,
    _remove_black_level as remove_blacklv,
    _wb_correction as wb_correction,
    _colour_correction as colour_correction,
    _unsharp as unsharp_masking,
    _gaussian_denoise as gaussian_denoising,
    _adjust_gamma as adjust_gamma,
)

# Pick one sample image for the walkthrough
demo_idx = list(sample_images_raw.keys())[0]
demo_raw = sample_images_raw[demo_idx].copy()
demo_label = classes[labels[demo_idx]]
max_val = 2**BITS - 1

print(f"Demo image: {os.path.basename(image_paths[demo_idx])}")
print(f"Class     : {demo_label} ({CLASS_FULL_NAMES.get(demo_label, '')})")
print(f"Shape     : {demo_raw.shape}")
print(f"Range     : [{demo_raw.min():.2f}, {demo_raw.max():.2f}]")

# Normalize to [0, 1]
img_norm = demo_raw / max_val

# --- Stage-by-stage processing ---
stages = {}

# 0. Normalised raw (Bayer mosaic)
stages['0. Raw Bayer\n(normalised)'] = img_norm.copy()

# 1. Black-level removal
img_bl = remove_blacklv(img_norm, BLACK_LEVEL)
stages['1. Black-level\nremoved'] = img_bl.copy()

# 2. Demosaicing
img_demo = demosaicing_CFA_Bayer_bilinear(img_bl)
stages['2. Demosaiced\n(bilinear)'] = img_demo.copy()

# 3. White balance
img_wb = wb_correction(img_demo, WHITE_BALANCE)
stages['3. White\nbalance'] = np.clip(img_wb, 0, 1)

# 4. Colour correction
img_cc = colour_correction(img_wb, COLOUR_MATRIX)
p99 = np.percentile(img_cc, 99)
if p99 > 1.0:
    img_cc = img_cc / p99
img_cc = np.clip(img_cc, 0, 1)
stages['4. Colour\ncorrection'] = img_cc.copy()

# 5. Sharpening (unsharp masking on full RGB)
img_sharp = unsharp_masking(img_cc, radius=1.0, amount=1.0)
stages['5. Sharpening\n(unsharp mask)'] = np.clip(img_sharp, 0, 1)

# 6. Denoising
img_denoise = gaussian_denoising(np.clip(img_sharp, 0, 1), sigma=0.5)
stages['6. Denoising\n(Gaussian)'] = np.clip(img_denoise, 0, 1)

# 7. Gamma correction
img_gamma = adjust_gamma(np.clip(img_denoise, 0, 1), gamma=2.2)
stages['7. Gamma\ncorrection'] = np.clip(img_gamma, 0, 1)

# --- Visualize all stages ---
n_stages = len(stages)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, (title, img) in enumerate(stages.items()):
    ax = axes[i]
    if img.ndim == 2:
        ax.imshow(img, cmap='gray', vmin=0, vmax=img.max())
    else:
        ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    # Add range annotation
    ax.text(0.02, 0.98, f'[{img.min():.3f}, {img.max():.3f}]',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Hide last axis if unused
for i in range(n_stages, len(axes)):
    axes[i].axis('off')

plt.suptitle(f'ISP Pipeline Walkthrough - {demo_label} ({CLASS_FULL_NAMES.get(demo_label, "")})',
             fontsize=18, y=1.01)
plt.tight_layout()
savefig(fig, 'isp_walkthrough')
plt.show()


## 6. ISP Configuration Comparison

Compare the visual effect of different debayer / sharpening / denoising combinations.

In [ ]:
# Compare different ISP configurations on the same raw image
configs = [
    {'debayer': 'bilinear',  'sharpening': 'unsharp_masking',   'denoising': 'gaussian_denoising'},
    {'debayer': 'malvar2004','sharpening': 'unsharp_masking',   'denoising': 'gaussian_denoising'},
    {'debayer': 'menon2007', 'sharpening': 'unsharp_masking',   'denoising': 'gaussian_denoising'},
    {'debayer': 'bilinear',  'sharpening': 'sharpening_filter', 'denoising': 'median_denoising'},
    {'debayer': 'bilinear',  'sharpening': 'unsharp_masking',   'denoising': 'fft_denoising'},
    {'debayer': 'bilinear',  'sharpening': 'unsharp_masking',   'denoising': 'none'},
]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, cfg in enumerate(configs):
    pipeline = RawProcessingPipeline(CAMERA_PARAMETERS, **cfg)
    result = pipeline(img_norm.copy())  # returns (C, H, W) tensor
    result_np = result.permute(1, 2, 0).numpy()
    result_np = np.clip(result_np, 0, 1)

    ax = axes[i]
    ax.imshow(result_np)
    title = f"D={cfg['debayer']}\nS={cfg['sharpening']}\nN={cfg['denoising']}"
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.suptitle(f'ISP Configuration Comparison - {demo_label}', fontsize=18, y=1.01)
plt.tight_layout()
savefig(fig, 'isp_config_comparison')
plt.show()


## 7. Per-channel statistics after ISP processing

In [ ]:
# Process a subset of images through the default ISP pipeline and compute per-channel stats
default_pipeline = RawProcessingPipeline(
    CAMERA_PARAMETERS,
    debayer='bilinear',
    sharpening='unsharp_masking',
    denoising='gaussian_denoising'
)

N_STATS = min(100, len(image_paths))
channel_means = {'R': [], 'G': [], 'B': []}
channel_stds  = {'R': [], 'G': [], 'B': []}
processed_sample = None

for idx in sample_idxs[:N_STATS]:
    raw = load_raw_image(image_paths[idx]) / max_val
    processed = default_pipeline(raw)  # (C, H, W) tensor
    for c, name in enumerate(['R', 'G', 'B']):
        ch = processed[c].numpy()
        channel_means[name].append(ch.mean())
        channel_stds[name].append(ch.std())
    if processed_sample is None:
        processed_sample = processed

# Statistics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_ch = {name: CHANNEL_COLORS[name] for name in ['R', 'G', 'B']}

for i, (ch_name, ax) in enumerate(zip(['R', 'G', 'B'], axes)):
    means = channel_means[ch_name]
    stds = channel_stds[ch_name]
    ax.hist(means, bins=30, color=colors_ch[ch_name], alpha=0.7, edgecolor=REPORT_EDGE, label='Mean')
    ax.axvline(np.mean(means), ls='--', color=REPORT_EDGE, label=f'mu={np.mean(means):.4f}')
    ax.set_xlabel(f'{ch_name} channel mean')
    ax.set_ylabel('Count')
    ax.set_title(f'{ch_name} channel - mu={np.mean(means):.4f}, sigma={np.mean(stds):.4f}')
    ax.legend()

plt.suptitle(f'Per-channel statistics after ISP ({N_STATS} images)', fontsize=18, y=1.02)
plt.tight_layout()
savefig(fig, 'per_channel_stats')
plt.show()

# Print summary table
print(f"\n{'Channel':<10}  {'Mean':>10}  {'Std':>10}  {'Dataset Mean':>14}  {'Dataset Std':>13}")
print('-' * 62)
for c, name in enumerate(['R', 'G', 'B']):
    m = np.mean(channel_means[name])
    s = np.mean(channel_stds[name])
    print(f"{name:<10}  {m:>10.4f}  {s:>10.4f}  {DATASET_MEAN[c]:>14.4f}  {DATASET_STD[c]:>13.4f}")


### Per-channel histogram of a processed sample

In [ ]:
# Visualise the per-channel histogram of one processed image
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Processed image
ax = axes[0]
img_show = processed_sample.permute(1, 2, 0).numpy()
img_show = np.clip(img_show, 0, 1)
ax.imshow(img_show)
ax.set_title('Processed image (after ISP)', fontsize=15, fontweight='bold')
ax.axis('off')

# Per-channel histogram
ax = axes[1]
for c, (name, color) in enumerate(zip(['R', 'G', 'B'], [CHANNEL_COLORS['R'], CHANNEL_COLORS['G'], CHANNEL_COLORS['B']])):
    channel_data = processed_sample[c].numpy().ravel()
    ax.hist(channel_data, bins=200, color=color, alpha=0.5, label=name, density=True)
ax.set_xlabel('Pixel value')
ax.set_ylabel('Density')
ax.set_title('Per-channel intensity distribution (ISP output)')
ax.legend()

plt.tight_layout()
savefig(fig, 'per_channel_histogram_sample')
plt.show()


## 8. PyTorch Dataset for RAW images (with ISP pipeline)

In [ ]:
class MicroscopyRAWDataset(Dataset):
    """
    PyTorch Dataset for the Microscopy RAW Bayer images.

    Loads 16-bit single-channel TIFF images (256×256), normalizes to [0, 1],
    and optionally applies an ISP pipeline (RawProcessingPipeline) to produce
    3-channel RGB tensors (C, H, W).
    """

    # Camera parameters
    black_level = [9.834368023181512e-06] * 4
    white_balance = [-0.6567, 1.9673, 3.5304]
    colour_matrix = [-2.0338, 0.0933, 0.4157,
                     -0.0286, 2.6464, -0.0574,
                     -0.5516, -0.0947, 2.9308]
    camera_parameters = (black_level, white_balance, colour_matrix)

    # Statistics after default ISP
    dataset_mean = [0.91, 0.83, 0.94]
    dataset_std  = [0.08, 0.13, 0.05]

    def __init__(self, image_paths, labels, classes, bits=16,
                 isp_pipeline=None, transform=None, cache=True):
        self.image_paths = image_paths
        self.labels = labels
        self.classes = classes
        self.bits = bits
        self.isp_pipeline = isp_pipeline
        self.transform = transform
        self._use_cache = cache
        self._cache = {}

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        label = self.labels[idx]

        if self._use_cache and idx in self._cache:
            img = self._cache[idx]
        else:
            raw = tifffile.imread(self.image_paths[idx])
            img = np.array(raw, dtype=np.float32)
            img = img / (2**self.bits - 1)  # normalize to [0, 1]

            if self.isp_pipeline is not None:
                img = self.isp_pipeline(img)  # returns (C, H, W) torch tensor
            else:
                # Return raw as single-channel tensor
                img = torch.from_numpy(img).unsqueeze(0).float()

            if self._use_cache:
                self._cache[idx] = img

        if self.transform is not None:
            img = self.transform(img)

        return img, label

    def __repr__(self):
        has_isp = self.isp_pipeline is not None
        return (f"MicroscopyRAWDataset(n_images={len(self)}, "
                f"n_classes={len(self.classes)}, isp={has_isp}, "
                f"classes={self.classes})")


# Create dataset with default ISP pipeline
isp = RawProcessingPipeline(
    CAMERA_PARAMETERS,
    debayer='bilinear',
    sharpening='unsharp_masking',
    denoising='gaussian_denoising'
)

dataset = MicroscopyRAWDataset(
    image_paths=list(image_paths),
    labels=labels,
    classes=classes,
    bits=BITS,
    isp_pipeline=isp,
    cache=False,
)

print(dataset)
print(f"\nLoading first sample...")
img, lbl = dataset[0]
print(f"  Shape : {img.shape}")
print(f"  Dtype : {img.dtype}")
print(f"  Range : [{img.min():.4f}, {img.max():.4f}]")
print(f"  Label : {lbl} ({classes[lbl]})")


## 9. Sample visualization per class (RAW -> ISP -> RGB)

In [ ]:
# Show RAW (Bayer) and processed RGB side by side, one per class
n_classes = len(classes)
fig, axes = plt.subplots(n_classes, 3, figsize=(12, 3.5 * n_classes))

# Find first index per class
class_first_idx = {}
for i, lbl in enumerate(labels):
    if lbl not in class_first_idx:
        class_first_idx[lbl] = i

for row, cls_idx in enumerate(range(n_classes)):
    idx = class_first_idx.get(cls_idx, 0)

    # Raw image
    raw_img = load_raw_image(image_paths[idx])
    raw_norm = raw_img / max_val

    # Processed image
    processed, _ = dataset[idx]
    proc_np = processed.permute(1, 2, 0).numpy()
    proc_np = np.clip(proc_np, 0, 1)

    # Col 0: RAW Bayer
    ax = axes[row, 0]
    ax.imshow(raw_norm, cmap='gray', vmin=0, vmax=raw_norm.max())
    if row == 0:
        ax.set_title('RAW Bayer', fontsize=14, fontweight='bold')
    ax.set_ylabel(f'{classes[cls_idx]}\n{CLASS_FULL_NAMES.get(classes[cls_idx], "")}',
                  fontsize=12, fontweight='bold', rotation=0, labelpad=110, va='center')
    ax.axis('off')

    # Col 1: Processed RGB
    ax = axes[row, 1]
    ax.imshow(proc_np)
    if row == 0:
        ax.set_title('After ISP (RGB)', fontsize=14, fontweight='bold')
    ax.axis('off')

    # Col 2: Per-channel histogram of processed image
    ax = axes[row, 2]
    for c, (name, color) in enumerate(zip(['R', 'G', 'B'], [CHANNEL_COLORS['R'], CHANNEL_COLORS['G'], CHANNEL_COLORS['B']])):
        ax.hist(processed[c].numpy().ravel(), bins=100, color=color, alpha=0.4, density=True, label=name)
    if row == 0:
        ax.set_title('Channel histograms', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlim(0, 1.2)

plt.suptitle('One sample per class - RAW -> ISP pipeline -> RGB', fontsize=18, y=1.0)
plt.tight_layout()
savefig(fig, 'samples_per_class')
plt.show()


## 10. Stratified Train / Validation split & DataLoader

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# --- Stratified Train / Val split ---
SEED = 42
TRAIN_SIZE = 0.7
BATCH_SIZE = 32

y = np.array(labels)
sss = StratifiedShuffleSplit(n_splits=1, train_size=TRAIN_SIZE, random_state=SEED)
train_idx, val_idx = next(sss.split(np.zeros(len(y)), y))

train_dataset = torch.utils.data.Subset(dataset, train_idx.tolist())
val_dataset   = torch.utils.data.Subset(dataset, val_idx.tolist())

print(f"Train : {len(train_dataset)} images ({TRAIN_SIZE*100:.0f}%)")
print(f"Val   : {len(val_dataset)} images ({(1-TRAIN_SIZE)*100:.0f}%)")

# Verify stratification
train_labels = [labels[i] for i in train_idx]
val_labels   = [labels[i] for i in val_idx]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, split_labels, title in [
    (axes[0], train_labels, f'Train ({len(train_labels)})'),
    (axes[1], val_labels, f'Validation ({len(val_labels)})')
]:
    split_str = [classes[l] for l in split_labels]
    split_counts = Counter(split_str)
    sorted_cls = sorted(split_counts.keys())
    vals = [split_counts[c] for c in sorted_cls]
    ax.bar(sorted_cls, vals, color=colors_for_labels(sorted_cls), edgecolor=REPORT_EDGE)
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.set_title(f'{title} - class distribution')
    ax.tick_params(axis='x', rotation=45)
    for bar, v in zip(ax.patches, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                str(v), ha='center', fontsize=10)

plt.suptitle('Stratified split verification', fontsize=18, y=1.02)
plt.tight_layout()
savefig(fig, 'stratified_split')
plt.show()

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Test batch
batch_imgs, batch_labels = next(iter(train_loader))
print(f"\nBatch shape  : {batch_imgs.shape}")
print(f"Batch dtype  : {batch_imgs.dtype}")
print(f"Batch range  : [{batch_imgs.min():.4f}, {batch_imgs.max():.4f}]")
print(f"Batch labels : {batch_labels.tolist()[:10]}...")


## 11. Training batch grid visualization

In [ ]:
import torchvision.utils as vutils

# Grab a batch from the train loader
batch_imgs, batch_labels = next(iter(train_loader))

n_show = min(16, batch_imgs.shape[0])
grid = vutils.make_grid(batch_imgs[:n_show], nrow=4, normalize=True, padding=2)
grid_np = grid.permute(1, 2, 0).numpy()

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(np.clip(grid_np, 0, 1))
ax.set_title('Training batch - Microscopy RAW (after ISP)', fontsize=18, fontweight='bold')
ax.axis('off')
plt.tight_layout()
savefig(fig, 'training_batch_grid')
plt.show()

print("Labels:", [classes[l] for l in batch_labels[:n_show].tolist()])


## 12. Bayer pattern analysis

Inspect the RGGB Bayer mosaic structure of the raw images.

In [ ]:
# Extract RGGB channels from the Bayer mosaic
demo_raw_norm = demo_raw / max_val

R  = demo_raw_norm[0::2, 0::2]   # Red
G1 = demo_raw_norm[0::2, 1::2]   # Green (row 0)
G2 = demo_raw_norm[1::2, 0::2]   # Green (row 1)
B  = demo_raw_norm[1::2, 1::2]   # Blue

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Row 0: individual Bayer channels
for ax, ch, title, cmap in zip(
    axes[0],
    [R, (G1 + G2) / 2, B],
    ['R (even row, even col)', 'G (avg of G1+G2)', 'B (odd row, odd col)'],
    ['Reds', 'Greens', 'Blues']
):
    im = ax.imshow(ch, cmap=cmap)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

# Row 1: histograms per Bayer channel + zoomed Bayer pattern
ax = axes[1, 0]
for ch, name, color in [(R, 'R', CHANNEL_COLORS['R']), (G1, 'G1', CHANNEL_COLORS['G1']), (G2, 'G2', CHANNEL_COLORS['G2']), (B, 'B', CHANNEL_COLORS['B'])]:
    ax.hist(ch.ravel(), bins=150, color=color, alpha=0.5, label=name, density=True)
ax.set_xlabel('Normalised intensity')
ax.set_ylabel('Density')
ax.set_title('Bayer channel distributions')
ax.legend()

# Zoomed Bayer mosaic (8x8 patch)
ax = axes[1, 1]
patch = demo_raw_norm[:8, :8]
ax.imshow(patch, cmap='gray', interpolation='nearest')
ax.set_title('Zoomed Bayer pattern (8x8)', fontsize=13, fontweight='bold')
for r in range(8):
    for c in range(8):
        if r % 2 == 0 and c % 2 == 0:
            color = CHANNEL_COLORS['R']
        elif r % 2 == 1 and c % 2 == 1:
            color = CHANNEL_COLORS['B']
        else:
            color = CHANNEL_COLORS['G']
        ax.add_patch(plt.Rectangle((c-0.5, r-0.5), 1, 1, fill=False, edgecolor=color, linewidth=2))
ax.axis('off')

# Channel statistics summary
ax = axes[1, 2]
ax.axis('off')
stats_text = "Bayer Channel Statistics\n" + "=" * 30
for ch_name, ch_data in [('R', R), ('G1', G1), ('G2', G2), ('B', B)]:
    stats_text += (f"\n{ch_name}: mean={ch_data.mean():.4f}, "
                   f"std={ch_data.std():.4f}, "
                   f"min={ch_data.min():.4f}, "
                   f"max={ch_data.max():.4f}")
stats_text += f"\n\nG1/G2 ratio: {G1.mean() / (G2.mean() + 1e-10):.4f}"
stats_text += f"\nR/B ratio: {R.mean() / (B.mean() + 1e-10):.4f}"
ax.text(0.1, 0.5, stats_text, fontsize=12, family='monospace',
        transform=ax.transAxes, va='center',
        bbox=dict(boxstyle='round', facecolor='#F7F4EA', alpha=0.8))

plt.suptitle(f'Bayer Mosaic Analysis - {demo_label}', fontsize=18, y=1.01)
plt.tight_layout()
savefig(fig, 'bayer_analysis')
plt.show()


## 13. Dataset summary

| Parameter | Value |
|---|---|
| Image type | Single-channel RAW Bayer TIFF, stored as float data with effective 16-bit range |
| Resolution | 256 x 256 x 1 (raw) -> 256 x 256 x 3 (after ISP) |
| Bayer pattern | RGGB |
| Classification classes in this notebook | `BAS`, `EOS`, `KSC`, `LYA`, `LYT`, `MON`, `NGB`, `NGS` |
| QC-only / excluded label | `UNC` (`Unclassifiable`), inspected separately and not used as a target class |
| Other ignored labels | Unsupported `BAD`; below-threshold `MMZ`, `MYO` with the current `drop` strategy |
| Missing-image handling | `drop` selected annotations whose RAW files are absent after duplicate/corrupt cleanup |
| Normalisation (raw) | `img / (2^16 - 1)` -> approximately [0, 1] |
| ISP Pipeline | Black-level -> Demosaic -> WB -> CC -> Sharpen -> Denoise -> Gamma |
| Camera black level | `9.83e-06` (all channels) |
| White balance gains | `[-0.6567, 1.9673, 3.5304]` |
| Colour matrix | 3x3 CCM (see code) |
| Gamma | 2.2 |
| Default debayer | Bilinear |
| Default sharpening | Unsharp masking (radius=1, amount=1) |
| Default denoising | Gaussian (sigma=0.5) |
| Mean (after ISP) | `[0.91, 0.83, 0.94]` |
| Std (after ISP) | `[0.08, 0.13, 0.05]` |


## 14. RGB to Grayscale: which approach to use?

Several approaches can convert the processed images to a single channel:

| Approach | Description |
|---|---|
| Single channel (R/G/B) | Keep one channel only - G has the highest variance |
| Standard luminance | $Y = 0.299R + 0.587G + 0.114B$ (ITU-R BT.601 perceptual weights) |
| Simple average | $(R + G + B) / 3$ |
| PCA (1st component) | Projection onto the 1st eigenvector of the 3 channels |
| Direct RAW | Use raw Bayer image without ISP (already single-channel) |

Below is a quantitative comparison across multiple images.

In [ ]:
from sklearn.decomposition import PCA

# ---------------------------------------------------------------------------
# Conversion functions
# ---------------------------------------------------------------------------
def to_gray_channel(img_chw, channel=1):
    """Extract a single channel (default: G=1)."""
    return img_chw[channel].numpy()

def to_gray_luminance(img_chw):
    """ITU-R BT.601 luminance: Y = 0.299R + 0.587G + 0.114B"""
    r, g, b = img_chw[0].numpy(), img_chw[1].numpy(), img_chw[2].numpy()
    return 0.299 * r + 0.587 * g + 0.114 * b

def to_gray_mean(img_chw):
    """Simple channel average."""
    return img_chw.mean(dim=0).numpy()

def to_gray_pca(img_chw, pca_model=None):
    """Project onto the 1st principal component of the 3 channels.
    If pca_model is provided, use it; otherwise fit on this image."""
    pixels = img_chw.reshape(3, -1).numpy().T  # (N, 3)
    if pca_model is None:
        pca_model = PCA(n_components=1)
        pca_model.fit(pixels)
    proj = pca_model.transform(pixels).reshape(img_chw.shape[1], img_chw.shape[2])
    # Normalise to [0, 1]
    proj = (proj - proj.min()) / (proj.max() - proj.min() + 1e-10)
    return proj, pca_model

def to_gray_raw(raw_path, bits=16):
    """Load the raw Bayer image directly (already single-channel)."""
    raw = tifffile.imread(raw_path)
    return np.array(raw, dtype=np.float32) / (2**bits - 1)

# ---------------------------------------------------------------------------
# Fit a global PCA on a subset of images
# ---------------------------------------------------------------------------
N_PCA_FIT = min(50, len(image_paths))
pca_pixels = []
for idx in sample_idxs[:N_PCA_FIT]:
    img_t, _ = dataset[idx]  # (3, 256, 256)
    pca_pixels.append(img_t.reshape(3, -1).numpy().T)

pca_pixels = np.concatenate(pca_pixels, axis=0)  # (N*65536, 3)
global_pca = PCA(n_components=1)
global_pca.fit(pca_pixels)

print(f"PCA fitted on {N_PCA_FIT} images ({pca_pixels.shape[0]:,} pixels)")
print(f"  1st component: [{global_pca.components_[0][0]:.4f}, "
      f"{global_pca.components_[0][1]:.4f}, {global_pca.components_[0][2]:.4f}]")
print(f"  Explained variance ratio: {global_pca.explained_variance_ratio_[0]:.4f}")
del pca_pixels  # free memory

### Visual comparison on 3 samples from different classes

In [ ]:
# Pick 3 diverse classes for visual comparison
vis_classes = ['NGS', 'LYT', 'EOS']
vis_idxs = []
for target in vis_classes:
    target_idx = class_to_idx[target]
    for i, lbl in enumerate(labels):
        if lbl == target_idx:
            vis_idxs.append(i)
            break

methods = {
    'RGB (ISP)':     lambda img, path: np.clip(img.permute(1, 2, 0).numpy(), 0, 1),
    'R channel':     lambda img, path: to_gray_channel(img, 0),
    'G channel':     lambda img, path: to_gray_channel(img, 1),
    'B channel':     lambda img, path: to_gray_channel(img, 2),
    'Luminance\n(BT.601)': lambda img, path: to_gray_luminance(img),
    'Mean':          lambda img, path: to_gray_mean(img),
    'PCA (1st)':     lambda img, path: to_gray_pca(img, global_pca)[0],
    'RAW Bayer':     lambda img, path: to_gray_raw(path),
}

n_methods = len(methods)
n_samples = len(vis_idxs)

fig, axes = plt.subplots(n_samples, n_methods, figsize=(2.8 * n_methods, 3.2 * n_samples))

for row, idx in enumerate(vis_idxs):
    img_t, lbl = dataset[idx]
    raw_path = image_paths[idx]
    cls_name = classes[lbl]

    for col, (method_name, method_fn) in enumerate(methods.items()):
        ax = axes[row, col]
        result = method_fn(img_t, raw_path)

        if result.ndim == 3:  # RGB
            ax.imshow(result)
        else:
            ax.imshow(result, cmap='gray', vmin=0, vmax=1)

        if row == 0:
            ax.set_title(method_name, fontsize=12, fontweight='bold')
        if col == 0:
            ax.set_ylabel(cls_name, fontsize=14, fontweight='bold')
        ax.axis('off')

plt.suptitle('Grayscale conversion methods - visual comparison', fontsize=18, y=1.01)
plt.tight_layout()
savefig(fig, 'grayscale_comparison')
plt.show()
